# Feature engineering basics

Bins, ratios, interactions; impact on model AUC.


In [ ]:
# interaction features can help tree models less than linear ones
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
rng=np.random.default_rng(5); n=2500
age=rng.integers(18,70,n); income=rng.normal(50,20,n).clip(10,150)
visits=rng.integers(0,40,n)
y=((income<35)&(visits<5) | (age>55)&(visits<3)).astype(int)
base=pd.DataFrame({'age':age,'income':income.round(1),'visits':visits})
fe=base.copy()
fe['income_per_visit']=fe['income']/(fe['visits']+1)
fe['age_bin']=pd.cut(fe['age'], bins=[17,30,45,60,70], labels=False)
fe=pd.get_dummies(fe, columns=['age_bin'], drop_first=True)
for name,X in [('raw',base),('fe',fe)]:
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,random_state=5,stratify=y)
    m=LogisticRegression(max_iter=2000).fit(Xtr,ytr)
    print(name,'auc',round(roc_auc_score(yte,m.predict_proba(Xte)[:,1]),3))
